# ATLAS Wind scenario preprocessing

This notebook prepares CMIP6 wind scenario files before they are used in the Wind Atlas workflow.

It reads the original CMIP6 files for one wind component, harmonises coordinates, cuts the data to the selected area, optionally aggregates the data to daily means, and saves a clean NetCDF file.

Expected input variables are:

```text
uas    Eastward near surface wind component
vas    Northward near surface wind component
```

The cleaned output can be used by the following Wind Atlas steps, such as downscaling, scaling, conversion and plotting.


## 0. Load libraries

Run this cell first. It imports the packages used to read NetCDF files, manage folders, work with shapefiles and export the processed dataset.


In [1]:
from pathlib import Path
import warnings

import geopandas as gpd
import numpy as np
import xarray as xr
import rioxarray  # noqa: F401. Needed to activate the .rio accessor in xarray
from shapely.ops import unary_union

warnings.filterwarnings("ignore")


## 1. User settings

Edit only this cell for a standard run.

Use relative folders whenever possible. This keeps the notebook portable and avoids exposing local machine paths.

For CMIP6 wind scenarios, use `VARIABLE = "uas"` for the eastward wind component or `VARIABLE = "vas"` for the northward wind component.


In [2]:
# Country or region name used in the output file names.
COUNTRY = "chile"

# CMIP6 wind component to process.
# Use "uas" for eastward wind and "vas" for northward wind.
VARIABLE = "uas"

# Human readable long name, used only for documentation and file organisation.
VARIABLE_LONG_NAME = {
    "uas": "10m_wind_u_component",
    "vas": "10m_wind_v_component",
}[VARIABLE]

# CMIP6 model and experiment.
MODEL = "CNRM-ESM2-1"
EXPERIMENT = "ssp370"

# Temporal aggregation.
# Use "daymean" to convert sub daily data to daily means.
# Use None if the source files are already daily.
AGGREGATION_MODE = "daymean"

# Period automatically selected from the experiment.
# Historical uses 1985 to 2014. Scenarios use 2015 to 2100.
if EXPERIMENT == "historical":
    START = "1985-01"
    END = "2014-12"
else:
    START = "2015-01"
    END = "2100-12"

# Main project folder.
# Adapt this to your local project structure.
PROJECT_DIR = Path("../data")

# Folder containing the original CMIP6 NetCDF files.
# Expected structure:
# ../data/raw/cmip6/10m_u_component_of_wind/global/ssp370/CNRM-ESM2-1/*.nc
INPUT_DIR = PROJECT_DIR / "esgf_downloads" / MODEL / EXPERIMENT / VARIABLE 

# Folder where the processed NetCDF file will be saved.
OUTPUT_DIR = PROJECT_DIR / "processed" / VARIABLE_LONG_NAME / COUNTRY / MODEL / EXPERIMENT 

# Optional shapefile used only for the quality check plot.
# Set to None if no boundary shapefile is available.
BOUNDARY_SHAPEFILE = PROJECT_DIR / "shapefiles" / COUNTRY / f"REGIONES_v1.shp"

# Overwrite existing output files.
OVERWRITE = True


## 2. Area settings

The notebook uses a simple bounding box to cut the global CMIP6 files.

The format is:

```text
[north, west, south, east]
```

For Chile, the western limit includes islands. If you need only continental Chile, reduce the westward extent.


In [3]:
AREAS = {
    "bolivia": [-9.6, -69.8, -23.0, -57.4],
    "argentina": [-21.7, -73.6, -55.1, -53.5],
    "ecuador": [1.9, -92.0, -5.3, -75.1],
    "peru": [0.1, -81.5, -18.5, -68.5],
    "colombia": [15.9, -81.7, -5.1, -65.9],
    "chile": [-16.0, -112.0, -57.0, -65.0],
}

AREA_BBOX = AREAS[COUNTRY]

# Extra margin in degrees around the selected area.
# This is useful because climate model grids are coarse.
BBOX_BUFFER_DEG = 2.5


## 3. Helper functions

These functions standardise coordinates, select the time period, cut the data to the target area and save a clean NetCDF file.


In [4]:
def standardise_coordinate_names(ds):
    """Rename common latitude and longitude names to longitude and latitude."""
    rename_map = {}

    if "lon" in ds.coords:
        rename_map["lon"] = "longitude"
    if "lat" in ds.coords:
        rename_map["lat"] = "latitude"

    if rename_map:
        ds = ds.rename(rename_map)

    if "longitude" not in ds.coords or "latitude" not in ds.coords:
        raise ValueError(
            "The dataset must contain longitude and latitude coordinates. "
            "Check the input NetCDF files."
        )

    return ds


def roll_longitude_to_minus180_180(ds, lon_name="longitude"):
    """Convert longitudes from 0 to 360 degrees into minus 180 to 180 degrees."""
    ds = ds.assign_coords({lon_name: ((ds[lon_name] + 180) % 360) - 180})
    return ds.sortby(lon_name)


def ensure_epsg4326(ds):
    """Assign EPSG:4326 to the dataset if possible."""
    ds = ds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)

    if ds.rio.crs is None:
        ds = ds.rio.write_crs("EPSG:4326", inplace=False)

    return ds


def fix_coords(ds):
    """Standardise spatial coordinates and CRS."""
    ds = standardise_coordinate_names(ds)

    ds["longitude"] = np.round(ds["longitude"], 3)
    ds["latitude"] = np.round(ds["latitude"], 3)

    if float(ds["longitude"].min()) >= 0:
        ds = roll_longitude_to_minus180_180(ds)

    ds = ds.sortby("longitude")
    ds = ds.sortby("latitude")
    ds = ensure_epsg4326(ds)

    return ds


def preprocess_cmip(ds):
    """Prepare each CMIP6 file before open_mfdataset combines them."""
    ds = fix_coords(ds)

    drop_candidates = [
        "time_bounds",
        "time_bnds",
        "bounds_latitude",
        "bounds_longitude",
        "height",
        "spatial_ref",
    ]
    ds = ds.drop_vars([v for v in drop_candidates if v in ds.variables], errors="ignore")

    if EXPERIMENT == "historical":
        ds = ds.sel(time=slice("1985", "2014"))
    else:
        ds = ds.sel(time=slice("2015", "2100"))

    for coord_name in ["member_id", "dcpp_init_year"]:
        if coord_name in ds.coords or coord_name in ds.dims:
            ds = ds.mean(coord_name)

    return ds


def load_data(input_dir):
    """Load all NetCDF files from the selected input folder."""
    input_dir = Path(input_dir)
    files = sorted(input_dir.glob("*.nc"))

    if not files:
        raise FileNotFoundError(
            f"No NetCDF files found in {input_dir}. "
            "Check INPUT_DIR in the User settings cell."
        )

    print(f"Found {len(files)} NetCDF file(s).")
    print(f"Input folder: {input_dir}")

    ds = xr.open_mfdataset(
        files,
        preprocess=preprocess_cmip,
        combine="by_coords",
        chunks={"time": 365},
    )

    if VARIABLE not in ds.data_vars:
        available = list(ds.data_vars)
        raise ValueError(
            f"Variable {VARIABLE!r} was not found in the input files. "
            f"Available variables are: {available}"
        )

    return ds


def cut_to_bbox(ds, bbox, buffer_deg=0.0):
    """Cut the dataset to a bounding box with an optional buffer."""
    north, west, south, east = bbox

    lon_min = west - buffer_deg
    lon_max = east + buffer_deg
    lat_min = south - buffer_deg
    lat_max = north + buffer_deg

    ds = ds.sortby("latitude").sortby("longitude")
    ds_cut = ds.sel(
        longitude=slice(lon_min, lon_max),
        latitude=slice(lat_min, lat_max),
    )

    if ds_cut.sizes.get("latitude", 0) == 0 or ds_cut.sizes.get("longitude", 0) == 0:
        raise ValueError(
            "The selected bbox produced an empty dataset. "
            "Check AREA_BBOX and the longitude convention of the input files."
        )

    return ds_cut


def aggregate_daily_mean(ds):
    """Aggregate the dataset to daily means."""
    if "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})

    if "time" not in ds.coords:
        raise ValueError("The dataset has no time coordinate, so daily aggregation cannot be applied.")

    return ds.resample(time="1D").mean()


def save_netcdf_fast(ds, output_filename, overwrite=True):
    """Save an xarray Dataset or DataArray as NetCDF without compression."""
    output_filename = Path(output_filename)

    if output_filename.exists() and not overwrite:
        print(f"File already exists and OVERWRITE is False: {output_filename}")
        return

    output_filename.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(ds, xr.DataArray):
        if ds.name is None:
            ds.name = VARIABLE
        ds = ds.to_dataset()
    elif not isinstance(ds, xr.Dataset):
        raise TypeError("Input must be an xarray Dataset or DataArray.")

    ds = ds.drop_vars("spatial_ref", errors="ignore")

    encoding = {}
    for data_var in ds.data_vars:
        enc = {"zlib": False}

        if np.issubdtype(ds[data_var].dtype, np.floating):
            enc["dtype"] = "float32"

        if ds[data_var].ndim >= 2:
            enc["chunksizes"] = tuple(min(size, 512) for size in ds[data_var].shape)

        encoding[data_var] = enc

    ds.to_netcdf(
        output_filename,
        engine="netcdf4",
        format="NETCDF4",
        encoding=encoding,
    )

    print(f"Data written to: {output_filename}")


def load_boundary(boundary_path):
    """Load a boundary shapefile if it exists."""
    if boundary_path is None:
        return None

    boundary_path = Path(boundary_path)

    if not boundary_path.exists():
        print(f"Boundary shapefile not found: {boundary_path}")
        return None

    boundary = gpd.read_file(boundary_path)

    if boundary.crs is None:
        boundary = boundary.set_crs("EPSG:4326")
    else:
        boundary = boundary.to_crs("EPSG:4326")

    return boundary


## 4. Load and preprocess the CMIP6 data

This cell loads the selected input files, cuts the global domain to the selected area and applies the requested daily aggregation.


In [5]:
print("Starting CMIP6 wind scenario preprocessing...")
print(f"Country: {COUNTRY}")
print(f"Variable: {VARIABLE}")
print(f"Model: {MODEL}")
print(f"Experiment: {EXPERIMENT}")
print(f"Period: {START} to {END}")

xdf = load_data(INPUT_DIR)
print("Dataset loaded.")

xdf = cut_to_bbox(xdf, AREA_BBOX, buffer_deg=BBOX_BUFFER_DEG)
print("Dataset cut to the selected area.")

if AGGREGATION_MODE == "daymean":
    xdf = aggregate_daily_mean(xdf)
    print("Data aggregated to daily means.")
elif AGGREGATION_MODE is None:
    print("No temporal aggregation applied.")
else:
    raise ValueError("AGGREGATION_MODE must be 'daymean' or None.")

xdf


Starting CMIP6 wind scenario preprocessing...
Country: chile
Variable: uas
Model: CNRM-ESM2-1
Experiment: ssp370
Period: 2015-01 to 2100-12
Found 1 NetCDF file(s).
Input folder: ../data/esgf_downloads/CNRM-ESM2-1/ssp370/uas
Dataset loaded.
Dataset cut to the selected area.


IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


Data aggregated to daily means.


<xarray.Dataset> Size: 149MB
Dimensions:    (time: 31411, latitude: 32, longitude: 37)
Coordinates:
  * latitude   (latitude) float64 256B -58.13 -56.73 -55.33 ... -16.11 -14.71
  * longitude  (longitude) float64 296B -113.9 -112.5 -111.1 ... -64.69 -63.28
  * time       (time) datetime64[ns] 251kB 2015-01-01 2015-01-02 ... 2100-12-31
Data variables:
    uas        (time, latitude, longitude) float32 149MB dask.array<chunksize=(1, 32, 37), meta=np.ndarray>
Attributes: (12/52)
    Conventions:            CF-1.7 CMIP-6.2
    creation_date:          2019-09-25T15:06:59Z
    description:            Future scenario with high radiative forcing by th...
    title:                  CNRM-ESM2-1 model output prepared for CMIP6 / Sce...
    activity_id:            ScenarioMIP AerChemMIP
    contact:                contact.cmip@meteo.fr
    ...                     ...
    nemo_gelato_commit:     49095b3accd5d4c_6524fe19b00467a
    arpege_minor_version:   6.3.2
    branch_time_in_parent:  60265.0
    branch_time_in_child:   60265.0
    history:                none
    tracking_id:            hdl:21.14100/bcc946be-1e97-458e-93ca-37195cf4c0fa

## 5. Save the processed NetCDF file

The output file name contains the variable, period, country, experiment and model.

This makes the file easy to recognise in the next steps of the workflow.


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_filename = OUTPUT_DIR / f"{VARIABLE}_{START}_{END}_{COUNTRY}_{EXPERIMENT}_{MODEL}_processed.nc"

save_netcdf_fast(
    xdf,
    output_filename=output_filename,
    overwrite=OVERWRITE,
)


Data written to: ../data/processed/10m_wind_u_component/chile/CNRM-ESM2-1/ssp370/uas_2015-01_2100-12_chile_ssp370_CNRM-ESM2-1_processed.nc


## 6. Quick quality check

This cell reopens the saved file and prints its structure.

Use it to confirm that the file was written correctly and that the selected variable is present.


In [7]:
test = xr.open_dataset(output_filename)
test


<xarray.Dataset> Size: 149MB
Dimensions:    (time: 31411, latitude: 32, longitude: 37)
Coordinates:
  * latitude   (latitude) float64 256B -58.13 -56.73 -55.33 ... -16.11 -14.71
  * longitude  (longitude) float64 296B -113.9 -112.5 -111.1 ... -64.69 -63.28
  * time       (time) datetime64[ns] 251kB 2015-01-01 2015-01-02 ... 2100-12-31
Data variables:
    uas        (time, latitude, longitude) float32 149MB ...
Attributes: (12/52)
    Conventions:            CF-1.7 CMIP-6.2
    creation_date:          2019-09-25T15:06:59Z
    description:            Future scenario with high radiative forcing by th...
    title:                  CNRM-ESM2-1 model output prepared for CMIP6 / Sce...
    activity_id:            ScenarioMIP AerChemMIP
    contact:                contact.cmip@meteo.fr
    ...                     ...
    nemo_gelato_commit:     49095b3accd5d4c_6524fe19b00467a
    arpege_minor_version:   6.3.2
    branch_time_in_parent:  60265.0
    branch_time_in_child:   60265.0
    history:                none
    tracking_id:            hdl:21.14100/bcc946be-1e97-458e-93ca-37195cf4c0fa

## 7. Optional map preview

This section plots the first available time step of the processed variable.

The boundary shapefile is optional. If it is not available, the notebook will still show the data map.


In [8]:
def plot_example(da, boundary=None):
    """Plot one time step of the processed variable with an optional boundary."""
    import matplotlib.pyplot as plt

    da = da.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
    da = da.rio.write_crs("EPSG:4326", inplace=False)

    fig, ax = plt.subplots(figsize=(9, 7))
    da.plot(ax=ax, cmap="viridis")

    if boundary is not None:
        boundary.boundary.plot(ax=ax, color="black", linewidth=0.8)

    ax.set_title(f"{VARIABLE} preview, first time step")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.show()


#boundary = load_boundary(BOUNDARY_SHAPEFILE)
#plot_example(test[VARIABLE].isel(time=0), boundary=boundary)


## 8. Notes for the next workflow step

The processed file created by this notebook is ready to be used as input for the following Wind Atlas notebooks.

Typical order:

```text
1. Scenario preprocessing
2. Downscaling
3. Scaling or bias correction
4. Wind component conversion
5. Plotting
```

Run this notebook once for `uas` and once for `vas` if both components are needed.
